In [ ]:
import pandas as pd

# 1. Import des données

In [ ]:
df_irve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/eb76d20a-8501-400e-b336-d85724de5435'
)

print(f"Shape : {df_irve.shape}")
df_irve.head()

Ce premier jeu de données est la base nationale des IRVE (Infrastructures de Recharge pour Véhicules Électriques).
Cette table comporte 210129 lignes pour 52 variables.

In [ ]:
df_ve = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/90e0d717-deda-4bdc-9987-f82faac5bc93',
 encoding='latin-1'
)

print(f"Shape : {df_ve.shape}")
df_ve.head()

Ce second jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 703545 lignes pour 8 variables.

In [ ]:
df_revenus = pd.read_csv(
 'https://www.data.gouv.fr/api/1/datasets/r/516130bc-4dcb-47f5-8347-ae96553c43ab',sep=';'
)

print(f"Shape : {df_revenus.shape}")
df_revenus.head()

Ce dernier jeu de données contient des informations sur les voitures particulières immatriculées par commune et par type de recharge.
Cette table comporte 34926 lignes pour 57 variables.

# 2. Choix de la variable de jointure

In [ ]:
list(df_irve.columns)

In [ ]:
list(df_ve.columns)

In [ ]:
list(df_revenus.columns)

Nous remarquons que ces 3 jeu de données contiennent une variable indiquant le code géographique mais sous 3 noms différents :
- `code_insee_commune` pour df_irve
- `CODGEO` pour df_ve
- `Code géographique` pour df_revenus

# 3. Préparation de la variable de jointure

In [ ]:
from src.preparation_data import diagnostic_cle_jointure

In [ ]:
diagnostic_cle_jointure(df_irve,"code_insee_commune","df_irve")
diagnostic_cle_jointure(df_ve,"CODGEO","df_ve")
diagnostic_cle_jointure(df_revenus,"Code géographique","df_revenus")

Ce diagnostic met en évidence plusieurs points d’attention :

- Présence de valeurs manquantes dans `code_insee_commune` (df_irve)  
- Présence de longueurs de codes incohérentes :
  - `code_insee_commune` (df_irve) contient des codes de longueur 3, 6 et 7
  - `CODGEO` (df_ve) contient des codes de longueur 4
- Les variables de jointure ne sont pas uniques :
  - `code_insee_commune` (df_irve)
  - `CODGEO` (df_ve)

Les clés de jointure présentent des problèmes de qualité et de format.  
Il est nécessaire de les nettoyer et les uniformiser (type, format, gestion des valeurs manquantes, unicité) avant d’effectuer la jointure.

#### a. repérons les problèmes de longueurs de codes incohérentes

In [ ]:
taile_louche = df_irve[
    df_irve["code_insee_commune"].notna() &
    (df_irve["code_insee_commune"].astype(str).str.len() == 7)
]
taile_louche["code_insee_commune"].head()

Les valeurs des `code_insee_commune`de longueur 3 sont : "0.0".
Les valeurs des `code_insee_commune`de longueur 6 sont les codes à 4 chiffres suivis de ".0".
Les valeurs des `code_insee_commune`de longueur 7 sont les codes à 5 chiffres suivis de ".0".

In [ ]:
taile_louche = df_ve[
    df_ve["CODGEO"].notna() &
    (df_ve["CODGEO"].astype(str).str.len() == 4)
]
taile_louche["CODGEO"].head()

Les valeurs des `code_insee_commune`de longueur 4 sont les codes à 4 chiffres des départements 1 (01) à 9 (09). Il faut rajouter un "0" au début du code.

#### b. uniformisons ces codes géographiques pour qu'ils soient tous à 5 chiffres

In [ ]:
from src.preparation_data import nettoyer_code_insee

In [ ]:
df_irve["code_insee_commune"] = df_irve["code_insee_commune"].apply(nettoyer_code_insee)
df_ve["CODGEO"] = df_ve["CODGEO"].apply(nettoyer_code_insee)
df_revenus["Code géographique"] = df_revenus["Code géographique"].apply(nettoyer_code_insee)

In [ ]:
diagnostic_cle_jointure(df_irve,"code_insee_commune","df_irve")
diagnostic_cle_jointure(df_ve,"CODGEO","df_ve")

#### c. diminuons le nombre de valeurs manquantes en utilisant la latitude et longitude

In [ ]:
from src.preparation_data import creer_gdf_irve, charger_communes, joindre_communes, ajouter_codes_geo

In [ ]:
gdf_irve = creer_gdf_irve(df_irve, "consolidated_longitude", "consolidated_latitude")
gdf_communes = charger_communes()
gdf_result = joindre_communes(gdf_irve, gdf_communes)
df_irve = ajouter_codes_geo(df_irve, gdf_result,"both")

Nous avons maintenant 3 colonnes concernant le code géographique dans la base de données irve :
- `code_insee_commune` : la variable initiale
- `code_geo_manquant` : la variable initiale complétée grace à la latitude et la longitude pour les valeurs manquantes
- `code_geo_total` : la variable recalculant tous les codes géographiques à partir de la latitude et la longitude, sans prendre en compte la variable initiale `code_insee_commune`

In [ ]:
from src.preparation_data import compter_valeurs_manquantes, compter_uniques

In [ ]:
colonnes_irve = ["code_insee_commune", "code_geo_manquant", "code_geo_total"]

# Valeurs manquantes
print("Valeurs manquantes :")
print(compter_valeurs_manquantes(df_irve, colonnes_irve))

# Valeurs uniques
print("\nValeurs uniques :")
print(compter_uniques(df_irve, colonnes_irve))

In [ ]:
set_irve_manquant = set(df_irve["code_geo_manquant"].dropna())
set_irve_total = set(df_irve["code_geo_total"].dropna())
set_ve = set(df_ve["CODGEO"].dropna())
set_revenus = set(df_revenus["Code géographique"].dropna())

In [ ]:
print("Différences avec VE :")
print(len(set_irve_manquant - set_ve))
print(len(set_irve_total - set_ve))

print("\nDifférences avec revenus :")
print(len(set_irve_manquant - set_revenus))
print(len(set_irve_total - set_revenus))

print("\nDifférences VE vs IRVE :")
print(len(set_ve - set_irve_manquant))
print(len(set_ve - set_irve_total))

print("\nDifférences revenus vs IRVE :")
print(len(set_revenus - set_irve_manquant))
print(len(set_revenus - set_irve_total))

In [ ]:
from src.preparation_data import corriger_codes_incoherents, corriger_par_nom, corriger_conflit_code_postal

In [ ]:
codes_manquants_communs = (set_irve_manquant - set_revenus) & (set_irve_manquant - set_ve)

print("Nombre de codes manquants dans les deux :")
print(len(codes_manquants_communs))

In [ ]:
df_irve = corriger_codes_incoherents(df_irve,codes_manquants_communs)

set_irve_manquant_final = set(df_irve["code_geo_manquant"].dropna())
nouveaux_manquants_communs = (set_irve_manquant_final - set_revenus) & (set_irve_manquant_final - set_ve)

print(f"Nombre de codes problématiques restants : {len(nouveaux_manquants_communs)}")

In [ ]:
df_irve = corriger_par_nom(df_irve, nouveaux_manquants_communs)

set_final = set(df_irve["code_geo_manquant"].dropna())
reste_a_corriger = (set_final - set_revenus) & (set_final - set_ve)
print(f"Nombre de codes problématiques restants après les deux étapes : {len(reste_a_corriger)}")

In [ ]:
df_irve["consolidated_code_postal"]=df_irve["consolidated_code_postal"].apply(nettoyer_code_insee)

vérifier que le code postal est bien de la même forme que le code insee

In [ ]:
corriger_conflit_code_postal(df_irve, reste_a_corriger)

set_anomalies_final = set(df_irve["code_geo_manquant"].dropna())
anomalies_finales = (set_anomalies_final - set_revenus) & (set_anomalies_final - set_ve)

print(f"Nombre final de codes géographiques non répertoriés : {len(anomalies_finales)}")


Pour les anomalies restantes je pense que c'est aussi le code postal qui est renseigné à la place du code géo, cela peut signifier que d'autres erreurs sont présentes dans "code_geo_manquant"...